In [ ]:
#pip install numpy pandas torch scikit-learn

  Using cached scikit_learn-1.8.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (2.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 29.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 31.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 MB 29.7 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.8 MB/s  0:00:00
Using cached scikit_learn-1.8.0-cp314-cp314-macosx_12_0_arm64.whl (8.1 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/

In [1]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

# Load and pre-process data

In [2]:
target_col = "log_average_ms"

feature_cols = ["conv_flops", "matmul_flops",
                  "elementwise_mb", "reduction_mb", "normalization_mb",
                  "movement_mb", "memory_mb", "l1d_cache_kb",
                  "l1i_cache_kb", "l2_cache_kb", "base_clock_mhz",
                  "num_cores", "memory_bandwith_gbs", 
                  "total_flops", "total_mb", "arithmetic_intensity",
                  "flops_per_core", "movement_frac_x_cores"]

metadata_cols = ["model", "cpu_provider",
                 "machine_type", "platform", "run_id"]

In [8]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [10]:
# load test, train, val sets

train_df = pd.read_csv("/content/drive/MyDrive/Data/train_set.csv")
val_df = pd.read_csv("/content/drive/MyDrive/Data/val_set.csv")
test_df = pd.read_csv("/content/drive/MyDrive/Data/test_set.csv")

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (100252, 154)
val: (21483, 154)
test: (21483, 154)


In [11]:
assert list(train_df.columns) == list(val_df.columns) == list(test_df.columns)

train_df.head()

,model,input_dimensions,input_dtypes,output_dimensions,output_dtypes,conv_flops,matmul_flops,elementwise_mb,reduction_mb,normalization_mb,...,base_clock_mhz,memory_bandwith_gbs,cpu_provider,machine_type,platform,repo_file,average_ms,stddev_ms,min_ms,max_ms
0,hardcorenas_d_Opset17_extended.onnx,x:1x3x224x224,x:float32,668:1x1000,668:float32,472440512,2560000,50.528229,5.204269,0.000000,...,2300.000,717,intel,xeon_plat,bluehive,hardcorenas_d_Opset17.onnx,14.069188,0.082284,13.931521,14.247209
1,vit_base_patch8_224_in21k_Opset17_disable_all....,x:1x3x224x224,x:float32,1089:1x21843,1089:float32,231211008,156097479168,1771.169711,0.000000,792.141724,...,2450.000,205,amd,epyc,gcloud,vit_base_patch8_224_in21k_Opset17.onnx,1481.456045,150.199862,1094.324125,1632.506909
2,tf_efficientnetv2_m_in21ft1k_Opset17_basic.onnx,x:1x3x384x384,x:float32,2466:1x1000,2466:float32,31470991360,2560000,1201.339462,67.736023,0.000000,...,2449.998,205,amd,epyc,gcloud,tf_efficientnetv2_m_in21ft1k_Opset17.onnx,301.127424,12.889699,286.464729,324.336829
3,shufflenet_v2_x1_5_Opset16.onnx,x:1x3x224x224,x:float32,1137:1x1000,1137:float32,266034528,2048000,8.246674,1.630875,0.000000,...,2449.998,205,amd,epyc,gcloud,shufflenet_v2_x1_5_Opset16.onnx,8.161246,0.036559,8.080030,8.197800
4,wide_resnet101_2_Opset16_timm.onnx,x:1x3x224x224,x:float32,971:1x1000,971:float32,45502005248,4096000,252.656250,4.218750,0.000000,...,2300.000,717,intel,xeon_plat,bluehive,wide_resnet101_2_Opset16_timm.onnx,38.147223,0.068437,38.035275,38.292409


In [12]:
# pre-processing function

def preprocess(df: pd.DataFrame, train=False) -> pd.DataFrame:

  #drop rows with missing features
  df = df.dropna(subset=["average_ms"])

  #remove rows with high standard deviation (only for train)
  cv_limit = 0.1 # 10% cv limit
  df["cv"] = df["stddev_ms"] / df["average_ms"]

  if train:
    df = df[df["cv"] <= cv_limit].copy()

  #add log_average_ms column
  df["log_average_ms"] = np.log(df["average_ms"])

  #remove decimals and round down
  for col in ["elementwise_mb", "reduction_mb", "normalization_mb", "movement_mb"]:
    df[col] = df[col].round(0)
  
  eps = 1e-9
  cores = df["num_cores"].clip(lower=1)
  
  df["total_flops"] = df["conv_flops"] + df["matmul_flops"]

  df["total_mb"] = (
      df["elementwise_mb"]
      + df["reduction_mb"]
      + df["movement_mb"]
  )

  df["arithmetic_intensity"] = df["total_flops"] / (df["total_mb"] + eps)

  df["flops_per_core"] = df["total_flops"] / cores

  df["movement_frac_x_cores"] = (df["movement_mb"] / (df["total_mb"] + eps)) * cores

  return df

In [13]:
train_df = preprocess(train_df, True)
test_df = preprocess(test_df, False)
val_df = preprocess(val_df, False)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (95814, 161)
val: (21483, 161)
test: (21483, 161)


# Model Training

In [ ]:
#pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 5.3 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [14]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [15]:
X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_val = val_df[feature_cols]
y_val = val_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]


In [16]:
def rmse_log(y_true, y_pred):
  return np.sqrt(mean_squared_error(y_true, y_pred))

In [17]:
def train_xgb(params, X_train, y_train, X_val, y_val):
  model = XGBRegressor(
      objective="reg:squarederror",
      tree_method="hist",
      eval_metric="rmse",
      random_state=67,
      n_jobs=-1,
      early_stopping_rounds=100,
      **params,
  )

  model.fit(
      X_train,
      y_train,
      eval_set=[(X_val, y_val)],
      verbose=100,
  )

  val_pred = model.predict(X_val)
  score = rmse_log(y_val, val_pred)

  return model, score

In [21]:
from sklearn.model_selection import ParameterSampler

param_dist = {
    "n_estimators": [500, 800, 1200, 1600, 2000, 2500, 3000, 4000, 5000],
    "learning_rate": [0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15],
    "max_depth": [3, 4, 5, 6, 7, 8, 10],
    "min_child_weight": [1, 2, 3, 5, 8, 10, 15, 20, 30, 50],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "reg_lambda": [0.0, 0.1, 0.5, 1.0, 3.0, 5.0, 10.0, 20.0, 50.0],
    "reg_alpha": [0.0, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0],
}

In [35]:
# sampled_params = list(ParameterSampler(
#     param_dist,
#     n_iter=60,
#     random_state=67,
# ))

# results = []

# best_model = None
# best_score = float("inf")
# best_params = None

for i, params in enumerate(sampled_params, start=58):
  model, val_rmse = train_xgb(params, X_train, y_train, X_val, y_val)

  row = {
        "run": i,
        "val_rmse_log": val_rmse,
        **params,
    }

  results.append(row)

  if val_rmse < best_score:
    best_score = val_rmse
    best_model = model
    best_params = params

  print(f"{i:02d} | val RMSE log = {val_rmse:.5f} | best = {best_score:.5f}")


tuning_results = pd.DataFrame(results).sort_values("val_rmse_log")
tuning_results.head(10)


[0]	validation_0-rmse:1.28454
[100]	validation_0-rmse:0.29113
[200]	validation_0-rmse:0.28974
[294]	validation_0-rmse:0.29066
58 | val RMSE log = 0.28974 | best = 0.28853
[0]	validation_0-rmse:1.43803
[100]	validation_0-rmse:0.35610
[200]	validation_0-rmse:0.30817
[300]	validation_0-rmse:0.29646
[400]	validation_0-rmse:0.29211
[500]	validation_0-rmse:0.29080
[600]	validation_0-rmse:0.29046
[700]	validation_0-rmse:0.29042
[778]	validation_0-rmse:0.29045
59 | val RMSE log = 0.29035 | best = 0.28853
[0]	validation_0-rmse:1.35953
[100]	validation_0-rmse:0.34018
[200]	validation_0-rmse:0.30628
[300]	validation_0-rmse:0.29755
[400]	validation_0-rmse:0.29430
[500]	validation_0-rmse:0.29304
[600]	validation_0-rmse:0.29220
[700]	validation_0-rmse:0.29154
[800]	validation_0-rmse:0.29153
[872]	validation_0-rmse:0.29169
60 | val RMSE log = 0.29141 | best = 0.28853
[0]	validation_0-rmse:1.30259
[100]	validation_0-rmse:0.31175
[200]	validation_0-rmse:0.29384
[300]	validation_0-rmse:0.29141
[400]	val

KeyboardInterrupt: 

# Evaluation

In [36]:
final_model = best_model

test_pred = final_model.predict(X_test)
test_rmse_log = np.sqrt(mean_squared_error(y_test, test_pred))

print("Test RMSE: ", test_rmse_log)

Test RMSE:  0.2834630431730758


In [37]:
def latency_metrics_from_log(y_log_true, y_log_pred):
    true_ms = np.exp(y_log_true)
    pred_ms = np.exp(y_log_pred)

    error_ms = pred_ms - true_ms
    rel_err = np.abs(error_ms) / true_ms
    ratio_err = np.maximum(pred_ms / true_ms, true_ms / pred_ms)

    rmse_log = np.sqrt(mean_squared_error(y_log_true, y_log_pred))
    rmse_ms = np.sqrt(np.mean(error_ms ** 2))
    rmse_percent = np.sqrt(np.mean(((error_ms / true_ms) * 100) ** 2))

    return {
        "rmse_log": rmse_log,
        "rmse_ms": rmse_ms,
        "rmse_percent": rmse_percent,

        "median_relative_error": np.median(rel_err),
        "p90_relative_error": np.percentile(rel_err, 90),
        "p95_relative_error": np.percentile(rel_err, 95),

        "median_percent_error": np.median(rel_err) * 100,
        "p90_percent_error": np.percentile(rel_err, 90) * 100,
        "p95_percent_error": np.percentile(rel_err, 95) * 100,

        "median_ratio_error": np.median(ratio_err),
        "p90_ratio_error": np.percentile(ratio_err, 90),

        "within_10pct": np.mean(rel_err <= 0.10),
        "within_25pct": np.mean(rel_err <= 0.25),
        "within_50pct": np.mean(rel_err <= 0.50),
        "within_2x": np.mean(ratio_err <= 2.0),
    }

In [38]:
val_metrics = latency_metrics_from_log(y_val, final_model.predict(X_val))
test_metrics = latency_metrics_from_log(y_test, final_model.predict(X_test))

pd.DataFrame([val_metrics, test_metrics], index=["val", "test"])

,rmse_log,rmse_ms,rmse_percent,median_relative_error,p90_relative_error,p95_relative_error,median_percent_error,p90_percent_error,p95_percent_error,median_ratio_error,p90_ratio_error,within_10pct,within_25pct,within_50pct,within_2x
val,0.288527,163.981771,39.200427,0.083199,0.420502,0.592877,8.319857,42.050171,59.287650,1.087672,1.527444,0.553880,0.807802,0.925476,0.958898
test,0.283463,148.513679,35.851394,0.082335,0.420670,0.596226,8.233537,42.066966,59.622642,1.086706,1.528139,0.558674,0.808314,0.925383,0.959270


# Analysis

In [27]:
test_results = test_df.copy()

test_results["pred_log_latency"] = test_pred
test_results["true_log_latency"] = y_test

test_results["pred_latency_ms"] = np.exp(test_results["pred_log_latency"])
test_results["true_latency_ms"] = np.exp(test_results["true_log_latency"])

test_results["relative_error"] = (
    np.abs(test_results["pred_latency_ms"] - test_results["true_latency_ms"])
    / test_results["true_latency_ms"]
)

In [28]:
def group_latency_metrics(g):
    y_true = g["true_log_latency"].to_numpy()
    y_pred = g["pred_log_latency"].to_numpy()

    true_ms = np.exp(y_true)
    pred_ms = np.exp(y_pred)

    rel_err = np.abs(pred_ms - true_ms) / true_ms

    return pd.Series({
        "count": len(g),
        "rmse_log": np.sqrt(mean_squared_error(y_true, y_pred)),
        "within_10pct": np.mean(rel_err <= 0.10),
        "within_25pct": np.mean(rel_err <= 0.25),
        "within_50pct": np.mean(rel_err <= 0.50),
    })

In [29]:
test_results.groupby("platform").apply(group_latency_metrics).reset_index()

/tmp/ipykernel_7223/3061731483.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_results.groupby("platform").apply(group_latency_metrics).reset_index()


,platform,count,rmse_log,within_10pct,within_25pct,within_50pct
0,bluehive,10744.0,0.326154,0.537323,0.771593,0.896314
1,gcloud,10739.0,0.233054,0.580035,0.845051,0.954465


In [31]:
test_results.groupby(["machine_type", "platform"]).apply(group_latency_metrics).reset_index()

/tmp/ipykernel_7223/2224414100.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_results.groupby(["machine_type", "platform"]).apply(group_latency_metrics).reset_index()


,machine_type,platform,count,rmse_log,within_10pct,within_25pct,within_50pct
0,epyc,gcloud,9388.0,0.205541,0.602365,0.866745,0.965168
1,xeon,gcloud,1351.0,0.371710,0.424870,0.694301,0.880089
2,xeon_plat,bluehive,10744.0,0.326154,0.537323,0.771593,0.896314


In [33]:
test_results.groupby(["cpu_provider", "platform"]).apply(group_latency_metrics).reset_index()

/tmp/ipykernel_7223/3658681541.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_results.groupby(["cpu_provider", "platform"]).apply(group_latency_metrics).reset_index()


,cpu_provider,platform,count,rmse_log,within_10pct,within_25pct,within_50pct
0,amd,gcloud,9388.0,0.205541,0.602365,0.866745,0.965168
1,intel,bluehive,10744.0,0.326154,0.537323,0.771593,0.896314
2,intel,gcloud,1351.0,0.371710,0.424870,0.694301,0.880089


In [34]:
test_results.groupby(["cpu_provider", "platform", "num_cores"]).apply(
    group_latency_metrics,
    include_groups=False
).reset_index()

,cpu_provider,platform,num_cores,count,rmse_log,within_10pct,within_25pct,within_50pct
0,amd,gcloud,1,2636.0,0.156167,0.725341,0.924127,0.979135
1,amd,gcloud,2,4015.0,0.196945,0.596015,0.876214,0.973848
2,amd,gcloud,4,2737.0,0.254011,0.493241,0.797589,0.938984
3,intel,bluehive,1,1347.0,0.240651,0.633259,0.864143,0.947290
4,intel,bluehive,2,2686.0,0.262453,0.627699,0.836932,0.929635
5,intel,bluehive,4,3999.0,0.319717,0.533133,0.767942,0.896224
6,intel,bluehive,8,2712.0,0.416789,0.406342,0.666298,0.838127
7,intel,gcloud,6,1351.0,0.371710,0.424870,0.694301,0.880089
